In [2]:
import skimage as sk
import os
import numpy as np
import pandas as pd
import re
from cellpose import models, io
from random import randint
from glob import glob

In [3]:
import napari

In [4]:
def clear_object_boarders(cell_masks,nuc_masks,name,path):
    cleared_cells, _, _ = sk.segmentation.relabel_sequential(sk.segmentation.clear_border(cell_masks))
    binary_cells = np.array(cleared_cells > 0,dtype="uint8")
    cleared_nuc, _, _ = sk.segmentation.relabel_sequential((binary_cells*nuc_masks))
    sk.io.imsave(os.path.join(path,f"{name}_cell_masks.tif"),cleared_cells,check_contrast=False)
    sk.io.imsave(os.path.join(path,f"{name}_nuclei_masks.tif"),cleared_nuc,check_contrast=False)
    return cleared_cells, cleared_nuc

def cyto_objects(cells, nuc, boundaries,name,path):
    binary_cells = np.array(cells>0,dtype='int8')
    binary_nuc = np.array(nuc>0,dtype='int8')
    erode_nuc = sk.morphology.erosion(binary_nuc,sk.morphology.footprint_rectangle((3,3)))
    #inter = (binary_cells - erode_nuc)
    binary_cyto = sk.exposure.rescale_intensity(((binary_cells - erode_nuc) - boundaries),in_range='image',out_range=(0,1)).astype(np.uint8)
    label_cyto = binary_cyto*cells
    sk.io.imsave(os.path.join(path,f"{name}_cyto_masks.tif"),label_cyto,check_contrast=False)
    return label_cyto

def get_measurements(label_cyto,label_cells,label_nuc,img,name):
    props = ["label","intensity_median"]
    reshape_img = np.moveaxis(img,0,-1)
    cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_img,properties=props))
    nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_nuc,reshape_img,properties=props))
    cell_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cells,reshape_img,properties=props))
    cyto_props = cyto_props.add_prefix("Cyto_")
    nuc_props = nuc_props.add_prefix("Nuc_")
    cell_props = cell_props.add_prefix("Cell_")
    cyto_props["Nur77_Nuc_Cyto_Ratio"] = nuc_props["Nuc_intensity_median-1"]/cyto_props["Cyto_intensity_median-1"]
    cyto_props["Image"] = name
    nuc_props["Image"] = name
    cell_props["Image"] = name
    return cyto_props, nuc_props, cell_props

def concat_measurements(dfs):
    return pd.concat(dfs,ignore_index=True)

In [5]:
files = sorted(glob("/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Exported_Tiffs/*.tif"))
names = [os.path.basename(f)[:-4] for f in files]

In [ ]:
model = models.CellposeModel(gpu=True)
masks_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Output_Masks/"
cyto_dfs = []
nuc_dfs = []
cell_dfs = []
for f, n in zip(files,names):
    avg_img = np.mean(sk.io.imread(f),axis=1)
    cell_masks, _, _ = model.eval(avg_img, normalize=True)
    nuc_masks, _, _ = model.eval(avg_img[0,...], normalize=True)
    object_boarders = sk.segmentation.find_boundaries(cell_masks,mode="inner")
    cleared_cells, cleared_nuc = clear_object_boarders(cell_masks,nuc_masks,n,masks_save_path)
    label_cyto = cyto_objects(cleared_cells, cleared_nuc, object_boarders,n,masks_save_path)
    cyto_props, nuc_props, cell_props = get_measurements(label_cyto,cleared_cells,cleared_nuc,avg_img,n)
    cyto_dfs.append(cyto_props)
    nuc_dfs.append(nuc_props)
    cell_dfs.append(cell_props)

df_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/"
merged_cyto = concat_measurements(cyto_dfs)
merged_cyto.to_csv(os.path.join(df_save_path,"Cytoplasm_Measurements.csv"))
merged_nuc = concat_measurements(nuc_dfs)
merged_nuc.to_csv(os.path.join(df_save_path,"Nuclei_Measurements.csv"))
merged_cell = concat_measurements(cell_dfs)
merged_cell.to_csv(os.path.join(df_save_path,"Cell_Measurements.csv"))


/home/gkristin/cellpose4/lib/python3.12/site-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


In [ ]:
i = randint(0,len(files))
img = sk.io.imread(files[i])
img.shape

In [ ]:
avg_img = np.mean(img,axis=1)
avg_img.shape

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(avg_img,channel_axis=0)

In [ ]:
model = models.CellposeModel(gpu=True)

In [ ]:
cell_masks, _, _ = model.eval(avg_img, normalize=True)
nuc_masks, _, _ = model.eval(avg_img[0,...], normalize=True)

In [ ]:
cell_masks, _, _ = model.eval(avg_img, normalize=True, rescale=0.5, resample=True, min_size=550)


In [ ]:
nuc_masks, _, _ = model.eval(avg_img[0,...], normalize=True, rescale=0.5, resample=True, min_size=550, cellprob_threshold=-1.0)

In [ ]:
v.add_labels(cell_masks)
v.add_labels(nuc_masks)

In [ ]:
object_boarders = sk.segmentation.find_boundaries(cell_masks,mode="inner")

In [ ]:
def clear_object_boarders(cell_masks,nuc_masks):
    cleared_cells, _, _ = sk.segmentation.relabel_sequential(sk.segmentation.clear_border(cell_masks))
    binary_cells = np.array(cleared_cells > 0,dtype="uint8")
    cleared_nuc, _, _ = sk.segmentation.relabel_sequential((binary_cells*nuc_masks))
    return cleared_cells, cleared_nuc

In [ ]:
def cyto_objects(cells, nuc, boundaries):
    binary_cells = np.array(cells>0,dtype='int8')
    binary_nuc = np.array(nuc>0,dtype='int8')
    erode_nuc = sk.morphology.erosion(binary_nuc,sk.morphology.footprint_rectangle((3,3)))
    #inter = (binary_cells - erode_nuc)
    binary_cyto = sk.exposure.rescale_intensity(((binary_cells - erode_nuc) - boundaries),in_range='image',out_range=(0,1)).astype(np.uint8)
    label_cyto = binary_cyto*cells
    return label_cyto

In [ ]:
def get_measurements(label_cyto,label_cells,label_nuc,img,name):
    props = ["label","intensity_median"]
    reshape_img = np.moveaxis(img,0,-1)
    cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_img,properties=props))
    nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_nuc,reshape_img,properties=props))
    cell_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cells,reshape_img,properties=props))
    cyto_props = cyto_props.add_prefix("Cyto_")
    nuc_props = nuc_props.add_prefix("Nuc_")
    cell_props = cell_props.add_prefix("Cell_")
    cyto_props["Nur77_Nuc_Cyto_Ratio"] = nuc_props["Nuc_intensity_median-1"]/cyto_props["Cyto_intensity_median-1"]
    cyto_props["Image"] = name
    nuc_props["Image"] = name
    cell_props["Image"] = name
    return cyto_props, nuc_props, cell_props

In [ ]:
def concat_measurements(dfs):
    return pd.concat(dfs,ignore_index=True)

In [ ]:
cleared_cells, cleared_nuc = clear_object_boarders(cell_masks,nuc_masks)

In [ ]:
label_cyto = cyto_objects(cleared_cells, cleared_nuc, object_boarders)

In [ ]:
v.add_labels(cleared_cells)
v.add_labels(cleared_nuc)
v.add_labels(label_cyto)

In [ ]:
df = get_measurements(label_cyto,cleared_cells,cleared_nuc,avg_img,names[i])

In [ ]:
df

In [ ]:
#measurements
props = ["label","intensity_median"]
reshape_avg = np.moveaxis(avg_img,0,-1)
cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_avg,properties=["label","intensity_median"]))
nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(nuc_masks,reshape_avg,properties=["label","intensity_median"]))


In [ ]:
cyto_props = cyto_props.add_prefix("Cyto_")
nuc_props = nuc_props.add_prefix("Nuc_")

In [ ]:
nuc_props.columns

In [ ]:
import seaborn as sns

In [ ]:
sns.swarmplot(nuc_props,x="Nuc_intensity_median-1",hue="Nuc_label")

In [ ]:
nuc_ints_map = sk.util.map_array(nuc_masks,nuc_props["label"],nuc_props["intensity_median-1"])

In [ ]:
v.add_image(nuc_ints_map)